LAB 1 PART A

In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from collections import deque
from typing import Any, Dict, Iterable, List, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

2) Abstract Problem Interface

In [2]:
class Problem(ABC):
    """Abstract base class for a search problem."""

    @abstractmethod
    def initial_state(self) -> Any:
        """Return the start state."""
        pass

    @abstractmethod
    def is_goal(self, state: Any) -> bool:
        """Return True if state is a goal state."""
        pass

    @abstractmethod
    def actions(self, state: Any) -> List[Any]:
        """Return the legal actions available in the given state."""
        pass

    @abstractmethod
    def result(self, state: Any, action: Any) -> Any:
        """Return the next state after applying action in state."""
        pass

    @abstractmethod
    def action_cost(self, state: Any, action: Any, next_state: Any) -> float:
        """Return the cost of applying action in state to reach next_state."""
        pass

3) Node Class

In [3]:
@dataclass
class Node:
    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1


@dataclass
class SearchResult:
    algorithm: str
    status: str
    solution: Optional[Node]
    nodes_expanded: int
    max_frontier_size: int
    reached_count: int = 0
    limit: Optional[int] = None
    iterations: Optional[List[Dict[str, Any]]] = None

    @property
    def path(self) -> Optional[List[Any]]:
        if self.solution is None:
            return None
        return reconstruct_path(self.solution)

    @property
    def solution_depth(self) -> Optional[int]:
        if self.solution is None:
            return None
        return self.solution.depth

    @property
    def solution_cost(self) -> Optional[float]:
        if self.solution is None:
            return None
        return self.solution.path_cost

4) Helper Functions

In [4]:
def reconstruct_path(node: Node) -> List[Any]:
    """Return the list of states from the root node to this node."""
    path = []

    while node is not None:
        path.append(node.state)
        node = node.parent

    path.reverse()
    return path


def reconstruct_actions(node: Node) -> List[Any]:
    """Return the list of actions from the root node to this node."""
    actions = []

    while node is not None and node.parent is not None:
        actions.append(node.action)
        node = node.parent

    actions.reverse()
    return actions


def state_is_on_path(node: Node, state: Any) -> bool:
    """
    Return True if state already appears on the path from the root to node.

    This is useful for depth-limited search because DLS often uses path-cycle
    checking instead of a global reached set.
    """
    while node is not None:
        if node.state == state:
            return True
        node = node.parent

    return False


def result_to_row(result: SearchResult) -> Dict[str, Any]:
    """Convert a SearchResult object into a row for a pandas DataFrame."""
    return {
        "Algorithm": result.algorithm,
        "Status": result.status,
        "Limit": result.limit,
        "Solution depth": result.solution_depth,
        "Solution cost": result.solution_cost,
        "Nodes expanded": result.nodes_expanded,
        "Max frontier/stack": result.max_frontier_size,
        "Reached states": result.reached_count,
    }


def show_results(results: List[SearchResult]) -> pd.DataFrame:
    """Display results as a DataFrame."""
    return pd.DataFrame([result_to_row(r) for r in results])

5) Implement the Grid Search Problem

In [ ]:
MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}


class GridProblem(Problem):
    def __init__(
        self,
        grid: List[List[int]],
        start: Tuple[int, int],
        goal: Tuple[int, int],
    ):
        """
        grid:
            2D list where 0 = free cell and 1 = obstacle.

        start, goal:
            Tuples in the form (row, col).
        """
        self.grid = grid
        self.start = start
        self.goal = goal

        self.rows = len(grid)
        self.cols = len(grid[0])

    def initial_state(self) -> Tuple[int, int]:
        return self.start

    def is_goal(self, state: Tuple[int, int]) -> bool:
        # TODO 1:
        # Return True if state is equal to the goal state.
        return state == self.goal
        raise NotImplementedError("Complete GridProblem.is_goal")

    def in_bounds(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return 0 <= row < self.rows and 0 <= col < self.cols

    def is_free(self, state: Tuple[int, int]) -> bool:
        row, col = state
        return self.grid[row][col] == 0

    def actions(self, state: Tuple[int, int]) -> List[str]:
        # TODO 2:
        # Return a list of legal action names.
        #
        # Steps:
        # 1. Create an empty list called legal_actions.
        legal_actions = []
        # 2. For each action in MOVES:
        # a. Compute the neighbour cell.
        row, col = state
        for action_type, (dr, dc) in MOVES.items():
            neighbour_state = (row + dr, col + dc)

            # b. Check that it is in bounds and c. Check that it is free.
            if self.in_bounds(neighbour_state) and self.is_free(neighbour_state):
                # d. If valid, add the action name to legal_actions.
                legal_actions.append(action_type)

        # 3. Return legal_actions.
        return legal_actions
        raise NotImplementedError("Complete GridProblem.actions")

    def result(self, state: Tuple[int, int], action: str) -> Tuple[int, int]:
        # TODO 3:
        # Return the next state after applying action to state.
        # Hint:
        # dr, dc = MOVES[action]
        row, col = state
        dr, dc = MOVES[action]
        # return (row + dr, col + dc)
        return (row + dr, col + dc)

    def action_cost(
        self,
        state: Tuple[int, int],
        action: str,
        next_state: Tuple[int, int],
    ) -> float:
        # TODO 4:
        # In this lab, each valid move has a cost of 1.
        # Return 1.
        return 1

Testing GridProblem

In [9]:
test_grid = [
    [0, 0, 0],
    [1, 1, 0],
    [0, 0, 0],
]

test_problem = GridProblem(test_grid, start=(0, 0), goal=(2, 2))

assert test_problem.initial_state() == (0, 0)
assert test_problem.is_goal((2, 2)) is True
assert test_problem.is_goal((0, 0)) is False
assert test_problem.actions((0, 0)) == ["RIGHT"]
assert test_problem.result((0, 0), "RIGHT") == (0, 1)
assert test_problem.action_cost((0, 0), "RIGHT", (0, 1)) == 1

print("GridProblem self-check passed.")

GridProblem self-check passed.


6. A Sample Drone Map

In [10]:
sample_grid = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0, 1, 0],
]

start = (0, 0)
goal = (9, 9)

problem = GridProblem(sample_grid, start, goal)

7. Visualising the Environment

In [11]:
def plot_path(
    grid: List[List[int]],
    start: Optional[Tuple[int, int]] = None,
    goal: Optional[Tuple[int, int]] = None,
    path: Optional[List[Tuple[int, int]]] = None,
    terrain_costs: Optional[List[List[float]]] = None,
    title: str = "Grid Map",
):
    """Visualise a grid and, optionally, a solution path."""
    arr = np.array(grid)
    height, width = arr.shape

    path_set = set(path) if path is not None else set()

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)

    for row in range(height):
        for col in range(width):
            state = (row, col)

            if arr[row, col] == 1:
                fill = (0.15, 0.15, 0.15)
            elif start is not None and state == start:
                fill = (0.95, 0.20, 0.20)
            elif goal is not None and state == goal:
                fill = (0.20, 0.70, 0.25)
            elif state in path_set:
                fill = (0.95, 0.90, 0.35)
            else:
                fill = (0.95, 0.95, 0.95)

            rect = patches.Rectangle(
                (col, row),
                1,
                1,
                linewidth=1,
                edgecolor=(0.75, 0.75, 0.75),
                facecolor=fill,
            )
            ax.add_patch(rect)

            if terrain_costs is not None and arr[row, col] == 0:
                ax.text(
                    col + 0.5,
                    row + 0.5,
                    str(terrain_costs[row][col]),
                    ha="center",
                    va="center",
                    fontsize=8,
                )

    plt.show()

In [ ]:
plot_path(sample_grid, start, goal, title="Sample Drone Map")

In [ ]:
class SearchAlgorithm(ABC):
    """Base class for search algorithms."""

    def expand(self, problem: Problem, node: Node) -> Iterable[Node]:
        # TODO 5:
        # Implement the AIMA-style EXPAND(problem, node).
        def expand(problem, node):
            # Pseudocode:
            # # s <- node.STATE
            s = node.state

            # for each action in problem.ACTIONS(s):
            for action in problem.actions(s):
                # s_prime <- problem.RESULT(s, action)
                s_prime = problem.result(s, action)
                # cost <- node.PATH_COST + problem.ACTION_COST(s, action, s_prime)
                cost = node.path_cost + problem.action_cost(s, action, s_prime)
                # yield NODE(STATE=s_prime, PARENT=node, ACTION=action, PATH_COST=cost)
                yield Node(state=s_prime, parent=node, action=action, path_cost=cost)

    @abstractmethod
    def search(self, problem: Problem) -> SearchResult:
        pass

In [ ]:
class BreadthFirstSearch(SearchAlgorithm):
    def search(self, problem: Problem) -> SearchResult:
        algorithm = "BFS"
        nodes_expanded = 0
        max_frontier_size = 1

        # TODO 6:
        # Implement BFS graph search using a FIFO queue.
        #
        # Steps:
        # 1. Create the initial node from problem.initial_state().
        node = Node(problem.initial_state())
        # 2. If the initial state is the goal, return success.
        if problem.is_goal(node.state):
            return node
        # 3. Create a deque frontier and add the initial node.
        frontier = deque([node])
        # 4. Create a reached set and add the initial state.
        reached_states = {problem.initial_state}
        # 5. While frontier is not empty:
        while len(frontier) > 0:
            # a. pop from the LEFT of the deque.
            node = frontier.popleft()
            # b. increment nodes_expanded.
            nodes_expanded += 1
            # c. expand the node.
            for child in self.expand(problem, node):
                # d. for each child:
                child_state = node.state
                #   i. if child is goal, return success.
                if problem.is_goal(child_state):
                    return SearchResult(
                        algorithm=algorithm,
                        status="Success",
                        solution=child,
                        nodes_expanded=nodes_expanded,
                        max_frontier_size=max_frontier_size,
                        reached_count=len(reached_states),
                    )
                #   ii. if child.state is not in reached:
                if child_state not in reached_states:
                    #       add child.state to reached.
                    reached_states.add(node.state)
                    #       append child to frontier.
                    frontier.append(child)
                # e. update max_frontier_size.
                max_frontier_size = max(max_frontier_size, len(frontier))

        # 6. Return failure if no solution is found.
        #
        # Hint:
        # frontier = deque([node])
        # node = frontier.popleft()
        return SearchResult(
            algorithm=algorithm,
            status="failure",
            solution=None,
            nodes_expanded=nodes_expanded,
            max_frontier_size=max_frontier_size,
            reached_count=len(reached_states),
        )
        raise NotImplementedError("Complete BreadthFirstSearch.search")


10. Depth First Search

In [ ]:
class DepthFirstSearch(SearchAlgorithm):
    def search(self, problem: Problem) -> SearchResult:
        algorithm = "DFS"
        max_frontier_size = 0
        nodes_expanded = 0

        # TODO 7:
        # Implement DFS graph search using a stack.
        #
        # Steps:
        # 1. Create the initial node.
        node = Node(problem.initial_state())
        # 2. If the initial state is the goal, return success.
        if problem.is_goal(node.state):
            return node
        # 3. Use a Python list as the stack frontier.
        frontier = [node]
        # 4. Use a reached set.
        reached_states = {problem.initial_state()}
        # 5. While frontier is not empty:
        while len(frontier) > 0:
            #   a. pop from the end of the list.
            node = frontier.pop()
            #   b. increment nodes_expanded.
            nodes_expanded += 1
            #   c. expand the node.
            for child in self.expand(problem, node):
                child_state = child.state

                if problem.is_goal(child_state):
                    return SearchResult(
                        algorithm=algorithm,
                        status="Success",
                        solution=child,
                        nodes_expanded=nodes_expanded,
                        max_frontier_size=max_frontier_size,
                        reached_count=len(reached_states),
                    )

                if child_state not in reached_states:
                    reached_states.add(child_state)
                    # d. add unreached children to the stack.
                    frontier.append(child)
                    # e. update max_frontier_size.
                    max_frontier_size = max(max_frontier_size, len(frontier))

            return SearchResult(
                algorithm=algorithm,
                status="failure",
                solution=None,
                nodes_expanded=nodes_expanded,
                max_frontier_size=max_frontier_size,
                reached_count=len(reached_states),
            )
        #
        # Note:
        # If you want DFS to explore actions in the same order as MOVES,
        # you may need to reverse the children before pushing them.
        raise NotImplementedError("Complete DepthFirstSearch.search")

11) Depth-Limited Search (DLS)

In [ ]:
class DepthLimitedSearch(SearchAlgorithm):
    def search(self, problem: Problem, limit: int = 10) -> SearchResult:
        algorithm = "DLS"

        initial_node = Node(problem.initial_state())

        metrics = {
            "nodes_expanded": 0,
            "max_stack_size": 1,
        }

        solution, status = self._recursive_dls(
            problem=problem,
            node=initial_node,
            limit=limit,
            metrics=metrics,
            current_stack_size=1,
        )

        return SearchResult(
            algorithm=algorithm,
            status=status,
            solution=solution,
            nodes_expanded=metrics["nodes_expanded"],
            max_frontier_size=metrics["max_stack_size"],
            reached_count=0,
            limit=limit,
        )

    def _recursive_dls(
        self,
        problem: Problem,
        node: Node,
        limit: int,
        metrics: Dict[str, int],
        current_stack_size: int,
    ) -> Tuple[Optional[Node], str]:
        # TODO 8:
        # Implement recursive depth-limited search.
        #
        # Steps:
        # 1. If node.state is goal, return (node, "success").
        if problem.is_goal(node.state):
            return (node, "success")
        # 2. Else if node.depth >= limit, return (None, "cutoff").
        elif node.depth >= limit:
            return (None, "cutoff")
        # 3. Otherwise:
        else:
            # a. increment metrics["nodes_expanded"].
            metrics["nodes_expanded"] += 1
            # b. set cutoff_occurred = False.
            cutoff_ocurred = False
            # c. for each child in expand(problem, node):
            for child in self.expand(problem, node):
                # i. skip the child if child.state already appears on the current path.
                if child.state in node.path:
                    continue
                # ii. update max_stack_size.
                metrics["max_stack_size"] = max(
                    metrics["max_stack_size"], current_stack_size
                )
                # iii. recursively call _recursive_dls on the child.
                result, status = self._recursive_dls(node=child, problem=problem)
                # iv. if result is "success", return success immediately.
                if status == "success":
                    result(result, status="success")
                # v. if result is "cutoff", set cutoff_occurred = True.
                elif status == "cutoff":
                    cutoff_ocurred = True
            # d. after all children:
            # if cutoff_occurred, return (None, "cutoff")
            if cutoff_ocurred:
                return (None, "cutoff")
            # else return (None, "failure")
            else:
                return (None, "failure")

        raise NotImplementedError("Complete DepthLimitedSearch._recursive_dls")

12) Iterative Deepening Search (IDS)

In [ ]:
class IterativeDeepeningSearch(SearchAlgorithm):
    def search(self, problem: Problem, max_depth: int = 50) -> SearchResult:
        algorithm = "IDS"
        total_nodes_expanded = 0
        max_stack_size = 0

        # TODO 9:
        # Implement IDS by repeatedly running DLS from limit 0 to max_depth.
        #
        # Requirements:
        # 1. Keep a list called iteration_log.
        iteration_log = []
        # 2. Accumulate total nodes expanded across all DLS iterations.
        dls = DepthLimitedSearch()
        # 3. Track the maximum stack size seen in any DLS run.
        for limit in range(0, max_depth + 1):
            result = dls.search(problem, limit=limit)

            total_nodes_expanded += result.nodes_expanded
            max_stack_size = max(max_stack_size, result.max_frontier_size)
            # 4. If a DLS run returns success, return a SearchResult for IDS.
            if result.status == "success":
                return SearchResult(
                    algorithm=algorithm,
                    status="success",
                    solution=result.solution,
                    nodes_expanded=total_nodes_expanded,
                    max_frontier_size=max_stack_size,
                    reached_count=0,
                    limit=limit,
                    iterationd=iteration_log,
                )
            # 5. If a DLS run returns failure, IDS can stop early and return failure.
            elif result.status == "failure":
                return SearchResult(
                    algorithm=algorithm,
                    status="failure",
                    solution=None,
                    nodes_expanded=total_nodes_expanded,
                    max_frontier_size=max_stack_size,
                    reached_count=0,
                    limit=limit,
                    iterationd=iteration_log,
                )

        # 6. If all limits return cutoff up to max_depth, return cutoff.
        #
        # Hint:
        # dls = DepthLimitedSearch()
        # result = dls.search(problem, limit=limit)
        return SearchResult(
            algorithm=algorithm,
            status="cutoff",
            solution=None,
            nodes_expanded=total_nodes_expanded,
            max_frontier_size=max_stack_size,
            reached_count=0,
            limit=max_depth,
            iterationd=iteration_log,
        )

        raise NotImplementedError("Complete IterativeDeepeningSearch.search")

13) Running the Algorithms on the Sample Map

In [19]:
bfs = BreadthFirstSearch()
dfs = DepthFirstSearch()
dls = DepthLimitedSearch()
ids = IterativeDeepeningSearch()

results = [
    bfs.search(problem),
    dfs.search(problem),
    dls.search(problem, limit=10),
    ids.search(problem, max_depth=30),
]

show_results(results)

TypeError: 'NoneType' object is not iterable

In [ ]:
# Visualise solution paths.
# After your algorithms work, choose at least two algorithms and plot their paths.

bfs_result = results[0]
dfs_result = results[1]

plot_path(
    sample_grid,
    start,
    goal,
    path=bfs_result.path,
    title="BFS Solution Path",
)

plot_path(
    sample_grid,
    start,
    goal,
    path=dfs_result.path,
    title="DFS Solution Path",
)

14) Creating My Own Maps

In [ ]:
# TODO 10:
# Create your first custom map here.

custom_grid_1 = [
    # Replace this with your own grid.
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [1, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 0, 1, 1],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [1, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 0, 0, 1, 1],
]

custom_start_1 = None
custom_goal_1 = None

# Example after completing:
# custom_problem_1 = GridProblem(custom_grid_1, custom_start_1, custom_goal_1)
# custom_results_1 = [
#     bfs.search(custom_problem_1),
#     dfs.search(custom_problem_1),
#     dls.search(custom_problem_1, limit=20),
#     ids.search(custom_problem_1, max_depth=40),
# ]
# show_results(custom_results_1)

In [ ]:
# TODO 11:
# Create your second custom map here.

custom_grid_2 = [
    # Replace this with your own grid.
]

custom_start_2 = None
custom_goal_2 = None

# Example after completing:
# custom_problem_2 = GridProblem(custom_grid_2, custom_start_2, custom_goal_2)
# custom_results_2 = [
#     bfs.search(custom_problem_2),
#     dfs.search(custom_problem_2),
#     dls.search(custom_problem_2, limit=20),
#     ids.search(custom_problem_2, max_depth=40),
# ]
# show_results(custom_results_2)

15) Reflection Questions

15.1 Problem Formulation

1) What is a state in this lab?
Ans: A cell in the grid representing a patch of land

2) What is an action?
Ans: That is anything that the drone can do at any point in time within an environment. In this case, the drone can move either left, right, up or down in the grid.

3) What does the result function do?
Ans: It serves as the transition function by returning the state the drone will be in after performing an action

4) Why is it useful to separate the problem definition from the search algorithm?
Ans: So that different types of search algorithms (various implementions) can be used on the problem with any problems

15.2 BFS
1) Why does BFS use a FIFO queue?
Ans: It ensures that all nodes at a particular level are expanded before the BFS goes deeper into one of the paths. 

2) Why does BFS find the shortest path in terms of number of steps on the unweighted grid?
Ans:

3) What role does the reached set play in BFS?

15.3 DFS
1) Why does DFS use a stack?
Ans: So that the algorithm picks a node, explores it before it backtracks to get the previous node at that level

2) Is DFS guaranteed to find the shortest path? Explain.
Ans:

3) Under what conditions can DFS use less memory than BFS?
Ans: When the nodes have a lot of children
4) Under what conditions can DFS perform badly?
Ans: Where there lots of possible states

15.4 DLS
1) What happens when the depth limit is too small?
2) What is the meaning of "cutoff"?
3) How is DLS different from ordinary DFS?
4) Why do we use path-cycle checking in DLS?

15.5 IDS
1) Why does IDS repeat DLS with increasing limits?
Ans: It safely finds you the shortest possible path with using much of memory

2) Why can IDS be complete even though DLS with a small limit is not?
Ans: 

3) Why does IDS use less memory than BFS?
4) What is the cost of repeatedly searching from the root?

15.6 Real-World Drone Context
1) In a real drone application, what might make one route safer or more practical than another?
Ans: The weather conditions and the lightning (visibility) in that portion of the terrain


2) Which algorithm would you choose if all moves are equally costly and you only care about the fewest number of moves? Explain.
Ans: If every move has the same cost, the total path cost is the number of moves. The BFS looks through the state space uniformly level by level, by the time it gets to the goal node, it is guaranteed to be the shortest possible path.

3) Which algorithm would you choose if you want to limit how deep the drone is allowed to search? Explain.
4) What limitations does this grid model have compared with real drone navigation?
Ans: Obstacles can span multiple cells and may not fully occupy the entire cell.(e.g. the obstacle spanning half of a cell). Also in real drone navigation, the drone can also move up, down and in any angle in any direction not just directly in one direction.